In [2]:
diretorio = r"C:\Users\Nicolas\Downloads\FARMA UTIL - 1 - 2024-07-26 16 55 00"
dfs = {}
contador = 0

In [3]:
import os

def listar_arquivos(directory_path):
    """
    Função para listar os arquivos em um diretório.

    :param directory_path: Caminho para o diretório a ser listado.
    :return: Lista de arquivos no diretório. Se o diretório não existir, retorna uma mensagem de erro.
    """
    try:
        # Verifica se o caminho é um diretório
        if not os.path.isdir(directory_path):
            return "O caminho fornecido não é um diretório ou não existe."
        
        # Lista os arquivos no diretório
        files = []
        for entry in os.listdir(directory_path):
            full_path = os.path.join(directory_path, entry)
            if os.path.isfile(full_path):
                files.append(entry)
        
        return files
    except Exception as e:
        return f"Ocorreu um erro: {str(e)}"


In [4]:

arquivos = listar_arquivos(diretorio)

In [54]:
arquivos

['374085.pdf']

In [7]:
import tabula
import re
import os
import cx_Oracle
import pandas as pd
from PyPDF2 import PdfReader
from openpyxl import Workbook, load_workbook
import base64
from pdf2image import convert_from_path
from PIL import Image
from io import BytesIO
import google.generativeai as genai
import PIL.Image
import tempfile

def conecta_oracle(sql):
        #Roda query para executar o Oracle
        dsn = cx_Oracle.makedsn("10.1.1.20", "1521", service_name="pdb1")
        user = "nicolas"
        password = "sinicolas"
        connection = cx_Oracle.connect(user, password, dsn)                       
        
        # Cria um cursor
        cursor = connection.cursor()
        # Executa uma consulta SQL
        cursor.execute(sql)
        tabela = cursor.fetchall()
        cursor.close()
        connection.close()

        return tabela
    
def obter_codigo(ean):
    sql = f"select cd_mercadoria||nr_dv_mercadoria from prddm.dc_mercadoria@rac where cd_ean_venda = '{ean}' and id_situacao_mercadoria = 'A'"
    tabela = conecta_oracle(sql=sql)

    if tabela:
        codigo = tabela[0]

        return codigo

    else:
        sql = f"""
        SELECT 
            VINC.CD_MERCADORIA, 
            MERC1.CD_EAN_VENDA AS EAN_MERCADORIA, 
            VINC.CD_MERCADORIA_VINCULADA, 
            MERC2.CD_EAN_VENDA AS EAN_MERCADORIA_VINCULADA
        FROM 
            PRDDM.DC_MERCADORIA_VINCULADA@RAC VINC
        LEFT JOIN 
            PRDDM.DC_MERCADORIA@RAC MERC1
            ON VINC.CD_MERCADORIA = MERC1.CD_MERCADORIA
        LEFT JOIN 
            PRDDM.DC_MERCADORIA@RAC MERC2
            ON VINC.CD_MERCADORIA_VINCULADA = MERC2.CD_MERCADORIA
            WHERE MERC1.CD_EAN_VENDA = '{ean}'
        """

        sql = f"select cd_mercadoria||nr_dv_mercadoria from prddm.dc_mercadoria@rac where cd_ean_venda = '{ean}' and id_situacao_mercadoria = 'A'"
        tabela = conecta_oracle(sql=sql)

        if tabela:
            codigo = tabela[0]

            return codigo

        else:
            return False


# Função para somar os valores de "VALOR TOTAL"
def sum_valor_total(documents):
    total_value = 0
    for doc in documents:
        total_value += float(doc['VALOR TOTAL'])
    return total_value


class ProcessamentoArquivo:

    def __init__(self, caminho_arquivo):
        self.caminho_arquivo = caminho_arquivo
        self.reader = PdfReader(caminho_arquivo)
        self.text = self._extrair_texto_pdf()
        self.lista_tabelas = self._ler_pdf_com_fallback()
        self.lista_retorno = []
        self.cnpjs, self.condicoes_pagamento, self.numero_pedido = self._extrair_informacoes_pdf()

    def _extrair_texto_pdf(self):
        text = ''
        for page in self.reader.pages:
            if page.extract_text():
                text += page.extract_text()
        return text            

    def _ler_pdf_com_fallback(self):
        try:
            lista_tabelas = tabula.read_pdf(self.caminho_arquivo, pages="all", multiple_tables=True)
            if not lista_tabelas or all(len(tabela) == 0 for tabela in lista_tabelas):
                raise ValueError("Tabelas não foram lidas corretamente")
            return lista_tabelas
        except:
            return self._extrair_tabela_manual()

    def _extrair_tabela_manual(self):
        text = self._extrair_texto_pdf()
        linhas = text.split('\n')
        tabelas = []
        tabela_atual = []
        for linha in linhas:
            if re.match(r'.*\d+.*', linha):
                tabela_atual.append(linha.split())
            if linha.strip() == "":
                if tabela_atual:
                    tabelas.append(pd.DataFrame(tabela_atual))
                    tabela_atual = []
        if tabela_atual:
            tabelas.append(pd.DataFrame(tabela_atual))
        return tabelas

    def _extrair_informacoes_pdf(self):
        sections = re.split(r'(Código:)', self.text)
        cnpjs = []
        condicoes_pagamento = []
        numero_pedido = ""
        
        for i in range(1, len(sections), 2):
            section_text = sections[i] + sections[i+1]
            cnpj_filial_match = re.search(r'CNPJ:\s*(\d{2}\.\d{3}\.\d{3}/\d{4}-\d{2})', section_text)
            condicao_pagamento_match = re.search(r'\n(.*?)Cotação:', section_text)
            numero_pedido_match = re.search(r'Cotação:\s*(\d+)', section_text)

            cnpj_filial = cnpj_filial_match.group(1) if cnpj_filial_match else ""
            condicao_pagamento = condicao_pagamento_match.group(1) if condicao_pagamento_match else ""
            if numero_pedido_match:
                numero_pedido = numero_pedido_match.group(1)
            condicao_pagamento = condicao_pagamento.replace("DIAS", "").strip()

            cnpjs.append(cnpj_filial)
            condicoes_pagamento.append(condicao_pagamento)

        return cnpjs, condicoes_pagamento, numero_pedido

    def processar_tabelas(self):
        contador_index = 0
        contador = 0
        qtd_itens = 0

        for index, tabela in enumerate(self.lista_tabelas):
            table = pd.DataFrame(tabela)
            quantidade_linhas = len(table)
            
            lista_ean_extraido = []
            lista_desconto = []
            lista_quantidade = []
            produto_encontrado = False

            if "Unidade:" in table.columns:
                contador_index -= 1

            ean = table.columns[0]
            valor_total = table.columns[3]
            desconto = table.columns[4]
            quantidade = table.columns[5]     

            if ("Produto" not in str(ean) or "Produto" not in str(desconto) or "Produto" not in str(quantidade)) and (ean != 0):
                pc_desconto = str(desconto)
                contador += 1
                total = float(str(valor_total).replace(",", ".")) * float(str(quantidade).replace(",", "."))
                cnpj = self.cnpjs[contador_index]
                cnpj = cnpj.replace("/","").replace("-","").replace(".", "").strip()
                ean = str(ean)
                desconto = str(desconto).replace("%", "").strip()
                quantidade = str(quantidade)
                valor_total = float(str(valor_total).replace(".", "").replace(",", "."))

                qtd_itens += 1
                
                codigo = obter_codigo(ean)

                if codigo:                            
                    dict_itens = {
                        "CODIGO": codigo[0],
                        "EAN": ean,
                        "O.C": self.numero_pedido,
                        "FILIAL": cnpj,
                        "DESCONTO": desconto,
                        "QUANTIDADE": quantidade,
                        "VALOR TOTAL": valor_total
                    }
                else:
                    dict_itens = {
                        "CODIGO": "Não encontrado",
                        "EAN": ean,
                        "O.C": self.numero_pedido,
                        "FILIAL": cnpj,
                        "DESCONTO": desconto,
                        "QUANTIDADE": quantidade,
                        "VALOR TOTAL": valor_total
                    }

                self.lista_retorno.append(dict_itens)
                lista_ean_extraido = []
                lista_desconto = []
                lista_quantidade = []
                lista_total = []
                produto_encontrado = True

            for linha in range(quantidade_linhas):
                qtd = len(table.values)

                try:
                    cod_ean = table.iloc[linha, 0]
                except:
                    cod_ean = None
                
                try:
                    desconto_1 = table.iloc[linha, 4]
                except:
                    desconto_1 = None

                try:
                    desconto_2 = table.iloc[linha, 5]
                except:
                    desconto_2 = None

                try:
                    quantidade_1 = table.iloc[linha, 5]
                except:
                    quantidade_1 = None

                try:
                    quantidade_2 = table.iloc[linha, 6]
                except:
                    quantidade_2 = None

                try:
                    quantidade_3 = table.iloc[linha, 7] if qtd > 5 else None
                except:
                    quantidade_3 = None

                try:
                    valor_total_item = table.iloc[linha, 6]
                except:
                    valor_total_item = None

                if not pd.isna(cod_ean):
                    try:
                        lista_ean_extraido.append(int(cod_ean))
                    except:
                        pass

                if not pd.isna(desconto_1) and "%" in str(desconto_1):
                    try:
                        lista_desconto.append(desconto_1)
                    except:
                        pass

                if not pd.isna(desconto_2) and "%" in str(desconto_2):
                    try:
                        lista_desconto.append(desconto_2)
                    except:
                        pass

                if not pd.isna(quantidade_1) and "," not in str(quantidade_1):
                    try:
                        lista_quantidade.append(int(quantidade_1))
                    except:
                        pass
                
                if not pd.isna(quantidade_2) and "," not in str(quantidade_2):
                    try:
                        lista_quantidade.append(int(quantidade_2))
                    except:
                        pass

                if not pd.isna(quantidade_3) and "," not in str(quantidade_3):
                    try:
                        lista_quantidade.append(int(quantidade_3))
                    except:
                        pass

                if not pd.isna(valor_total_item) and "," in str(valor_total_item):
                    try:
                        lista_total.append(float(str(valor_total_item).replace(",", ".")))
                    except:
                        pass

                if lista_ean_extraido and lista_desconto and lista_quantidade and lista_total:
                    pc_desconto = lista_desconto[0]
                    contador += 1
                    cnpj = self.cnpjs[contador_index]
                    cnpj = cnpj.replace("/","").replace("-","").replace(".", "").strip()
                    ean = lista_ean_extraido[0]
                    desconto = pc_desconto.replace("%", "").strip()
                    quantidade = lista_quantidade[0]
                    total = lista_total[0]

                    qtd_itens += 1
                    
                    codigo = obter_codigo(ean)

                    if codigo:                            
                        dict_itens = {
                            "CODIGO": codigo[0],
                            "EAN": ean,
                            "O.C": self.numero_pedido,
                            "FILIAL": cnpj,
                            "DESCONTO": desconto,
                            "QUANTIDADE": quantidade,
                            "VALOR TOTAL": total
                        }
                    else:
                        dict_itens = {
                            "CODIGO": "Não encontrado",
                            "EAN": ean,
                            "O.C": self.numero_pedido,
                            "FILIAL": cnpj,
                            "DESCONTO": desconto,
                            "QUANTIDADE": quantidade,
                            "VALOR TOTAL": total
                        }

                    self.lista_retorno.append(dict_itens)
                    lista_ean_extraido = []
                    lista_desconto = []
                    lista_quantidade = []
                    lista_total = []
                    produto_encontrado = True

            if produto_encontrado:
                contador_index += 1

        if produto_encontrado == False:
            item_anterior = ""
            lista_ean_extraido = []
            lista_quantidade = []
            lista_desconto = []
            lista_valor = []

            for valor in table.values:
                if "Unidade" not in valor and "Data" not in valor and "Cotação" not in valor[0] and "Totais" not in valor and "CNPJ" not in valor and "Página" not in valor and ":" not in valor[1]:
                    for item in valor:
                        if not pd.isna(item) and "," not in str(item):
                            try:
                                lista_quantidade.append(int(item))
                            except:
                                pass

                        if not pd.isna(item) and "%" in str(item):
                            try:
                                lista_desconto.append(item_anterior)
                            except:
                                pass

                        if not pd.isna(item):
                            match = re.search(r'(\d{8,13})', str(item))
                            if match:
                                ean = match.group(1)
                                try:
                                    lista_ean_extraido.append(int(ean))
                                except:
                                    pass  

                        if not pd.isna(item) and "," in str(item):
                            try:
                                lista_valor.append(int(item))
                            except:
                                pass

                        item_anterior = str(item)

                        if lista_ean_extraido and lista_desconto and lista_quantidade and lista_valor:
                            pc_desconto = lista_desconto[0]
                            contador += 1
                            cnpj = self.cnpjs[contador_index]
                            cnpj = cnpj.replace("/","").replace("-","").replace(".", "").strip()
                            ean = lista_ean_extraido[0]
                            desconto = pc_desconto.replace("%", "").strip()
                            quantidade = lista_quantidade[0]
                            valor = lista_valor[0]
                            valor = float(str(valor).replace(".", "").replace(",", "."))

                            qtd_itens += 1

                            codigo = obter_codigo(ean)

                            if codigo:                            
                                dict_itens = {
                                    "CODIGO": codigo[0],
                                    "EAN": ean,
                                    "O.C": self.numero_pedido,
                                    "FILIAL": cnpj,
                                    "DESCONTO": desconto,
                                    "QUANTIDADE": quantidade,
                                    "VALOR TOTAL": total
                                }
                            else:
                                dict_itens = {
                                    "CODIGO": "Não encontrado",
                                    "EAN": ean,
                                    "O.C": self.numero_pedido,
                                    "FILIAL": cnpj,
                                    "DESCONTO": desconto,
                                    "QUANTIDADE": quantidade,
                                    "VALOR TOTAL": total
                                }

                            self.lista_retorno.append(dict_itens)
                            lista_ean_extraido = []
                            lista_desconto = []
                            lista_quantidade = []
                            produto_encontrado = True

        return self.lista_retorno

    def exec(self):
        self._extrair_texto_pdf()
        self._extrair_informacoes_pdf()
        return self.processar_tabelas()

    def convert_pdf_to_jpg_base64(self, pdf_path):
        pages = convert_from_path(pdf_path, poppler_path=r'C:\poppler\bin')  # Atualize com o caminho do Poppler no Windows
        base64_images = []

        for page in pages:
            buffered = BytesIO()
            page.save(buffered, format="JPEG")
            img_str = base64.b64encode(buffered.getvalue()).decode('utf-8')
            base64_images.append(img_str)

        return base64_images

    def listar_arquivos(self, directory_path):
        """
        Função para listar os arquivos em um diretório.

        :param directory_path: Caminho para o diretório a ser listado.
        :return: Lista de arquivos no diretório. Se o diretório não existir, retorna uma mensagem de erro.
        """
        try:
            if not os.path.isdir(directory_path):
                return "O caminho fornecido não é um diretório ou não existe."
            
            files = []
            for entry in os.listdir(directory_path):
                full_path = os.path.join(directory_path, entry)
                if os.path.isfile(full_path):
                    files.append(entry)
            
            return files
        except Exception as e:
            return f"Ocorreu um erro: {str(e)}"

    def convert_pdf_to_images(self, pdf_path):
        temp_dir = tempfile.mkdtemp()
        images = convert_from_path(pdf_path, poppler_path=r'C:\poppler\bin')
        image_paths = []
        for i, image in enumerate(images):
            image_path = os.path.join(temp_dir, f"page_{i + 1}.jpg")
            image.save(image_path, 'JPEG')
            image_paths.append(image_path)
        return image_paths

    def executa_gemini(self, caminho_arquivo, prompt):
        """Executa análise de imagem usando o Gemini."""
        api_key = "'SECRET_REMOVED_BY_AI'"
        genai.configure(api_key=api_key)

        if caminho_arquivo.endswith(".pdf"):
            image_paths = self.convert_pdf_to_images(caminho_arquivo)
            img_path = image_paths[0]
            img = Image.open(img_path)
        else:
            img = PIL.Image.open(caminho_arquivo)

        generation_config = {
            "candidate_count": 1,
            "temperature": 0
        }
        safety_settings = {
            "HARASSMENT": "BLOCK_NONE",
            "HATE": "BLOCK_NONE",
            "SEXUAL": "BLOCK_NONE",
            "DANGEROUS": "BLOCK_NONE"
        }

        model = genai.GenerativeModel(
            model_name='gemini-1.5-flash',
            generation_config=generation_config,
            safety_settings=safety_settings
        )

        response = model.generate_content(
        [prompt, img], stream=True
    )
        response.resolve()

        if "|" in response.text:
            lista_retorno = []

            arquivo_dividido = response.text.split("|")

            all_items = []
            for item in arquivo_dividido:
                # Split adicional por "\n"
                all_items.extend(item.split("\n"))

            ean = None
            numero_pedido = None
            cnpj = None
            desconto = None
            quantidade = None
            total_pedido = None
            descricao = None
            qtd_itens = 0

            for item in all_items:
                if "EAN" in item:
                    ean = item.replace("EAN: ", "").strip()
                elif "Número" in item:
                    numero_pedido = item.replace("Número do pedido: ", "").replace(".", "").strip()
                elif "CNPJ" in item:
                    cnpj = item.replace("CNPJ: ", "").replace("/","").replace("-","").replace(".", "").strip()
                elif "Desconto" in item:
                    desconto = item.replace("Desconto: ", "").replace("%", "").replace(".", ",").strip()
                elif "Quantidade" in item:
                    quantidade = item.replace("Quantidade: ", "").replace(",", "").replace(".", "").strip()
                elif "Total do pedido" in item:
                    total_pedido = item.replace("Total do pedido: ", "").replace("R$", "").replace(".", "").replace(",", ".").strip()
                elif "Descrição" in item:
                    descricao = item.replace("Descrição do item: ", "").replace(",", "").replace(".", "").strip()

                if ean != None and numero_pedido != None and cnpj != None and desconto != None and quantidade != None and total_pedido != None and descricao != None:
                    pc_desconto = str(desconto)
                    cnpj = cnpj
                    cnpj = cnpj.replace("/","").replace("-","").replace(".", "").strip()
                    ean = str(ean)
                    desconto = str(desconto).replace("%", "").strip()
                    quantidade = str(quantidade)
                    total_pedido = float(total_pedido)

                    qtd_itens += 1
                    
                    codigo = obter_codigo(ean)

                    if codigo:                            
                        dict_itens = {
                            "CODIGO": codigo[0],
                            "EAN": ean,
                            "DESCRIÇÃO": descricao,
                            "O.C": self.numero_pedido,
                            "FILIAL": cnpj,
                            "DESCONTO": desconto,
                            "QUANTIDADE": quantidade,
                            "VALOR TOTAL": total_pedido
                        }
                    else:
                        dict_itens = {
                            "CODIGO": "Não encontrado",
                            "EAN": ean,
                            "DESCRIÇÃO": descricao,
                            "O.C": self.numero_pedido,
                            "FILIAL": cnpj,
                            "DESCONTO": desconto,
                            "QUANTIDADE": quantidade,
                            "VALOR TOTAL": total_pedido
                        }
                   
                    lista_retorno.append(dict_itens)

                    ean = None
                    numero_pedido = None
                    cnpj = None
                    desconto = None
                    quantidade = None
                    total_pedido = None
                    descricao = None

            return lista_retorno
        else:
            return False

    def processamento_arquivos(self):
        #df_limpo = self.exec()
        df_limpo = False

        if not df_limpo:
            prompt = f"Você é um especialista em leitura de imagens, vou te enviar um arquivo de um pedido, é composto por uma tabela com os itens, em que temos um cabeçalho com o número do pedido, cotação do cliente, CNPJ e, logo abaixo, uma tabela com os itens, quero que você me retorne em cada linha de EAN: EAN, número do pedido (ou cotação), CNPJ, desconto do item, a descrição dos itens, a quantidade de todos os itens cotados e o valor total de cada item. Um ponto importante: Essas informações devem formar uma linha a cada item da tabela, pode acontecer de o pedido estar em uma página e o restante, desse mesmo pedido, em outro, você deve perceber isso e fazer a unificação. O retorno que quero na sua saída é o seguinte: EAN: x | Número do pedido: x | CNPJ: x | Desconto: x | Descrição do item: x | Quantidade: x | Total do pedido: x (em formato BR, 0,00)"
            df_limpo = self.executa_gemini(self.caminho_arquivo, prompt)

        return df_limpo

In [8]:
# Inicializa o contador e o dicionário de DataFrames
contador = 0
dfs = {}

# Processa cada arquivo na lista de arquivos
for arquivo in arquivos:
    contador += 1
    processador = ProcessamentoArquivo(rf"{diretorio}\{arquivo}")
    df = processador.processamento_arquivos()
    print(arquivo)
    print(df)
    print(f"Qtd itens: {len(df)}")
    total_value = sum_valor_total(df)
    print(f"Valor total: {total_value}")
    dfs[contador] = df

374085.pdf
[{'CODIGO': '442951', 'EAN': '7898567750277', 'DESCRIÇÃO': 'DEPI ROLL FOLHAS PRONTAS FACIAL ARGAN C/8 PARES', 'O.C': '12362', 'FILIAL': '85035327000747', 'DESCONTO': '8,00', 'QUANTIDADE': '1', 'VALOR TOTAL': 10.56}, {'CODIGO': '533508', 'EAN': '7898567750666', 'DESCRIÇÃO': 'DEPI ROLL FOLHAS PRONTAS FACIAL COCONUT C/8 PARES', 'O.C': '12362', 'FILIAL': '85035327000747', 'DESCONTO': '8,00', 'QUANTIDADE': '1', 'VALOR TOTAL': 10.56}, {'CODIGO': '381074', 'EAN': '7898567750055', 'DESCRIÇÃO': 'DEPI ROLL FOLHAS PRONTAS FACIAL SPA CARE C/8 PARES', 'O.C': '12362', 'FILIAL': '85035327000747', 'DESCONTO': '8,00', 'QUANTIDADE': '1', 'VALOR TOTAL': 10.56}, {'CODIGO': '253205', 'EAN': '7896235353294', 'DESCRIÇÃO': 'LEITE DE COLONIA TEMPO AMAR C/200 ML', 'O.C': '12362', 'FILIAL': '85035327000747', 'DESCONTO': '10,00', 'QUANTIDADE': '3', 'VALOR TOTAL': 22.19}]
Qtd itens: 4
Valor total: 53.870000000000005
374092.pdf
[{'CODIGO': '699728', 'EAN': '7896049528604', 'DESCRIÇÃO': 'HERBISSIMO DES AN

In [57]:
dfs

{1: []}

In [19]:


# records = []
# for key, values in dfs.items():
#     print(values)
#     for value in values:
#         print(value)
#         value['KEY'] = key  # Adicionar a chave principal ao registro
#         records.append(value)

# # Criar o DataFrame
# df = pd.DataFrame(records)

374092.pdf
374128.pdf
374104.pdf
374125.pdf
374117.pdf
374136.pdf
374100.pdf
374113.pdf
374108.pdf
374140.pdf
374085.pdf
374094.pdf


In [14]:
df

,EAN,O.C,FILIAL,DESCONTO,QUANTIDADE,KEY
0,7899674027672,12377,85035327000313,"30,69",2,6
1,7896015525583,12377,85035327000313,"50,00",1,6
2,7896015528577,12377,85035327000313,"50,00",2,6
3,7897595903914,12377,85035327000313,"39,65",1,6
4,11509031310,12377,85035327000313,"15,00",4,6
5,11509031334,12377,85035327000313,"15,00",3,6
6,7896049528505,12377,85035327000313,"5,00",1,6
7,7896342415694,12377,85035327000313,"15,00",1,6
8,7891653014215,12377,85035327000313,"15,00",1,6


In [36]:
df.to_excel(rf"C:\Users\Rafael\Downloads\farma-util\arquivo_farma-util1.xlsx")

In [13]:
df